Install Dependencies

In [ ]:
%%bash
# Enables an effective pipeline implementation
pip install haystack-ai
pip install "sentence-transformers>=2.2.0" "huggingface_hub>=0.22.0"
pip install markdown-it-py mdit_plain pypdf
pip install gdown


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.0/380.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.0/298.0 kB 5.3 MB/s eta 0:00:00


Unzip the Dataset

In [ ]:
import zipfile
import os

# Path to the zipped file already uploaded to Colab
zip_file_path = 'EducationalDataFile.zip'

# Directory where the contents will be extracted
output_dir = '/content/Datafiles'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Unzipping the file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(output_dir)
    print(f"Files extracted to {output_dir}")

# List all files extracted:
for dirname, _, filenames in os.walk(output_dir):
    for filename in filenames:
        print(os.path.join(dirname, filename))


Files extracted to /content/Datafiles
/content/Datafiles/EducationalDataFile/Data Science Topics Questions.pdf
/content/Datafiles/EducationalDataFile/Computer Science Easy and Difficult Exam Questions.pdf


Import Necessary Components

In [ ]:
from haystack.components.writers import DocumentWriter
from haystack.components.converters import MarkdownToDocument, PyPDFToDocument, TextFileToDocument
from haystack.components.preprocessors import DocumentSplitter, DocumentCleaner
from haystack.components.routers import FileTypeRouter
from haystack.components.joiners import DocumentJoiner
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack import Pipeline
from haystack.document_stores.in_memory import InMemoryDocumentStore

# Initialize the in-memory document store
document_store = InMemoryDocumentStore()

# Set up the file type router to handle specific MIME types
file_type_router = FileTypeRouter(mime_types=["text/plain", "application/pdf", "text/markdown"])

# Initialize converters for different file formats
text_file_converter = TextFileToDocument()
markdown_converter = MarkdownToDocument()
pdf_converter = PyPDFToDocument()

# Initialize the document joiner component
document_joiner = DocumentJoiner()


Set Up Document Cleaner and Splitter

In [ ]:
document_cleaner = DocumentCleaner()
document_splitter = DocumentSplitter(split_by="word", split_length=150, split_overlap=50)


Initialize Document Embedder

In [ ]:
# Using a more advanced model for embedding documents
document_embedder = SentenceTransformersDocumentEmbedder(model="sentence-transformers/msmarco-roberta-base-v2")

# Initialize the document writer
document_writer = DocumentWriter(document_store)


Define the Processing Pipeline

In [ ]:
# Define the pipeline for document processing
preprocessing_pipeline = Pipeline()

# Adding components to the pipeline
preprocessing_pipeline.add_component(instance=file_type_router, name="file_type_router")
preprocessing_pipeline.add_component(instance=text_file_converter, name="text_file_converter")
preprocessing_pipeline.add_component(instance=markdown_converter, name="markdown_converter")
preprocessing_pipeline.add_component(instance=pdf_converter, name="pypdf_converter")
preprocessing_pipeline.add_component(instance=document_joiner, name="document_joiner")
preprocessing_pipeline.add_component(instance=document_cleaner, name="document_cleaner")
preprocessing_pipeline.add_component(instance=document_splitter, name="document_splitter")
preprocessing_pipeline.add_component(instance=document_embedder, name="document_embedder")
preprocessing_pipeline.add_component(instance=document_writer, name="document_writer")


Connect Pipeline Components

In [ ]:
# Connecting components in the pipeline
preprocessing_pipeline.connect("file_type_router.text/plain", "text_file_converter.sources")
preprocessing_pipeline.connect("file_type_router.application/pdf", "pypdf_converter.sources")
preprocessing_pipeline.connect("file_type_router.text/markdown", "markdown_converter.sources")

# Connect converters to the document joiner
preprocessing_pipeline.connect("text_file_converter", "document_joiner")
preprocessing_pipeline.connect("pypdf_converter", "document_joiner")
preprocessing_pipeline.connect("markdown_converter", "document_joiner")

# Connect the document joiner to subsequent components
preprocessing_pipeline.connect("document_joiner", "document_cleaner")
preprocessing_pipeline.connect("document_cleaner", "document_splitter")
preprocessing_pipeline.connect("document_splitter", "document_embedder")
preprocessing_pipeline.connect("document_embedder", "document_writer")


🚅 Components
  - file_type_router: FileTypeRouter
  - text_file_converter: TextFileToDocument
  - markdown_converter: MarkdownToDocument
  - pypdf_converter: PyPDFToDocument
  - document_joiner: DocumentJoiner
  - document_cleaner: DocumentCleaner
  - document_splitter: DocumentSplitter
  - document_embedder: SentenceTransformersDocumentEmbedder
  - document_writer: DocumentWriter
🛤️ Connections
  - file_type_router.text/plain -> text_file_converter.sources (List[Union[str, Path, ByteStream]])
  - file_type_router.application/pdf -> pypdf_converter.sources (List[Union[str, Path, ByteStream]])
  - file_type_router.text/markdown -> markdown_converter.sources (List[Union[str, Path, ByteStream]])
  - text_file_converter.documents -> document_joiner.documents (List[Document])
  - markdown_converter.documents -> document_joiner.documents (List[Document])
  - pypdf_converter.documents -> document_joiner.documents (List[Document])
  - document_joiner.documents -> document_cleaner.documents (Li

Run the Pipeline

In [ ]:
from pathlib import Path
preprocessing_pipeline.run({"file_type_router": {"sources": list(Path(output_dir).glob("**/*"))}})


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.73k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

{'file_type_router': {'unclassified': [PosixPath('/content/Datafiles/EducationalDataFile')]},
 'document_writer': {'documents_written': 101}}

Set Up Hugging Face API Token

In [ ]:
import os
from getpass import getpass

# Check if the Hugging Face API token is already set in the environment variables
if "HF_API_TOKEN" not in os.environ:
    # If not set, prompt the user to enter the Hugging Face API token securely
    os.environ["HF_API_TOKEN"] = getpass("Enter Hugging Face token:")


Enter Hugging Face token:··········


Define the Prompt Template and Set Up the Interactive UI

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import os
from haystack import Pipeline
from haystack.components.embedders import SentenceTransformersTextEmbedder
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack.components.builders import PromptBuilder
from haystack.components.generators import HuggingFaceAPIGenerator

# Template for the prompt, customized for generating exam questions
template = """
You are an advanced educational AI specialized in generating high-quality exam questions for Computer Science and Data Science. When a user provides a set of instructions or questions, generate exactly the number of questions requested, ensuring the following:

1. **Specificity and Relevance**:
   - Ensure each question is distinct and directly relates to the topic or subtopic provided by the user.
   - If the user does not specify a topic, choose relevant topics from the broader field of Computer Science and Data Science and clearly indicate the topics from which the questions were derived.

2. **Difficulty Calibration**:
   - Adhere strictly to the requested difficulty level:
     - Easy Questions: Maximum of 15 words.
     - Difficult Questions: 20 to 35 words.
     - Complex Questions: 25 to 35 words.
   - Ensure that the complexity and length of each question match the specified level.

3. **Question Variety**:
   - Introduce a variety of question formats
   - Each new question should cover a different aspect or concept within the chosen topic.

4. **Clarity and Precision**:
   - Ensure each question is clear, concise, and unambiguous.
   - Avoid redundancy by ensuring that each new question presents a unique challenge.

5. **Response Format**:
   - Provide exactly the number of questions requested, no more, no less.
   - Do not generate any follow-up questions, additional explanations, or other content not explicitly requested.

6. **Brevity**:
    - Discard any additional content that does not match the expected format.
    - Ensure that only the relevant content is shown, without any additional instructions or list items.

Context:
{% for document in documents %}
    {{ document.content }}
{% endfor %}

User's Request:
{{ question }}

Generated Questions:
"""

# Setup the pipeline
def setup_pipeline(api_key):
    os.environ["HF_API_TOKEN"] = api_key

    global pipe
    pipe = Pipeline()
    pipe.add_component("embedder", SentenceTransformersTextEmbedder(model="sentence-transformers/msmarco-roberta-base-v2"))
    pipe.add_component("retriever", InMemoryEmbeddingRetriever(document_store=document_store))
    pipe.add_component("prompt_builder", PromptBuilder(template=template))
    pipe.add_component(
        "llm",
        HuggingFaceAPIGenerator(api_type="serverless_inference_api", api_params={"model": "HuggingFaceH4/zephyr-7b-beta"}),
    )

    pipe.connect("embedder.embedding", "retriever.query_embedding")
    pipe.connect("retriever", "prompt_builder.documents")
    pipe.connect("prompt_builder", "llm")

# Define courses and corresponding topics
courses = {
    "Computer Science": [
        "Programming Languages",
        "Data Structures",
        "Algorithms",
        "Databases",
        "Operating Systems",
        "Computer Networks",
        "Software Engineering",
        "Cybersecurity",
        "Theory of Computation"
    ],
    "Data Science": [
        "Machine Learning",
        "Deep Learning",
        "Data Mining",
        "Statistics",
        "Data Visualization",
        "Natural Language Processing",
        "Big Data",
        "Data Ethics"
    ]
}

difficulty_options = ["Easy", "Difficult", "Complex"]

course_selector = widgets.Dropdown(
    options=list(courses.keys()),
    description='Course:',
    disabled=False
)

topic_selector = widgets.Dropdown(
    description='Topic:',
    disabled=False
)

def update_topics(change):
    selected_course = change['new']
    topic_selector.options = courses[selected_course]

course_selector.observe(update_topics, names='value')

difficulty_selector = widgets.Dropdown(
    options=difficulty_options,
    description='Difficulty:',
    disabled=False
)

question_number_input = widgets.Dropdown(
    options=[1, 2, 3, 4, 5],
    description='Number of Questions:',
    disabled=False
)

output = widgets.Output()

def on_submit_key(b):
    api_key = api_key_input.value
    setup_pipeline(api_key)
    clear_output()
    display(HTML("<h3>Welcome! Please select the course, topic, difficulty level, and number of questions you'd like to generate.</h3>"))
    display(course_selector)
    display(topic_selector)
    display(difficulty_selector)
    display(question_number_input)
    display(submit_button_generate)
    display(output)

def on_generate_questions(b):
    selected_course = course_selector.value
    selected_topic = topic_selector.value
    selected_difficulty = difficulty_selector.value
    number_of_questions = question_number_input.value

    user_input = f"Provide me with {number_of_questions} {selected_difficulty.lower()} questions from the topic of {selected_topic} while using RAG"

    with output:
        clear_output(wait=True)
        display(HTML(f"<div>Here are the {number_of_questions} questions you requested.</div>"))
        display(HTML(f'<div style="text-align: left; background-color: lightgreen; padding: 10px; border-radius: 5px; margin-top: 10px;">{user_input}</div>'))
        display(HTML("<i>Processing...</i>"))

        response = pipe.run({"text": user_input})

        # Ensure only the exact number of questions requested is displayed
        response_text = response["llm"]["replies"][0]
        response_lines = response_text.strip().split('\n')
        generated_questions = [line.strip().lstrip('12345. ') for line in response_lines if line.strip()]
        limited_questions = generated_questions[:number_of_questions]

        if len(limited_questions) < number_of_questions:
            display(HTML(f'<div style="color: red; margin-top: 10px;">Warning: The model generated only {len(limited_questions)} questions. Please try again or check your input.</div>'))

        # Display exactly the number of requested questions, without extra details
        for question in limited_questions:
            display(HTML(f'<div style="text-align: left; background-color: lightblue; padding: 10px; border-radius: 5px; margin-top: 10px;">{question}</div>'))

# API Key submission button
submit_button_key = widgets.Button(description="Submit API Key", button_style='success')
submit_button_key.on_click(on_submit_key)

# Question generation submission button
submit_button_generate = widgets.Button(description="Generate Questions", button_style='success', layout=widgets.Layout(width='auto'))
submit_button_generate.on_click(on_generate_questions)

# Initial UI for API key entry
api_key_input = widgets.Text(
    placeholder='Enter your Hugging Face API key here...',
    description='API Key:',
    disabled=False
)

display(api_key_input)
display(submit_button_key)


Dropdown(description='Course:', options=('Computer Science', 'Data Science'), value='Computer Science')

Dropdown(description='Topic:', options=(), value=None)

Dropdown(description='Difficulty:', options=('Easy', 'Difficult', 'Complex'), value='Easy')

Dropdown(description='Number of Questions:', options=(1, 2, 3, 4, 5), value=1)

Button(button_style='success', description='Generate Questions', layout=Layout(width='auto'), style=ButtonStyl…

Output()